# Multimodal Generative Models — Hands-on Notebook

<a target="_blank" href="https://colab.research.google.com/github/uitml/GenerativeAI_course/blob/main/tasks/multimodal_hands_on_notebook_worksheet.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Runtime recommendation
- **Best experience:** Colab with **GPU**
- **Works on CPU:** yes, but slower
- In Colab: `Runtime -> Change runtime type -> T4 GPU` if available

## What this notebook covers
1. CLIP for image–text similarity  
2. BLIP for image captioning  
3. BLIP for visual question answering  
4. Inspecting internal tensor shapes  
5. Guided reflection questions for students


This worksheet version also includes an **extended optional LLaVA demo** for stronger GPU runtimes.

## 0a. Colab-friendly install and runtime check

If you are in Colab, run the next cell.  
If you are in a local environment with everything already installed, you can skip it.


In [ ]:
# Colab / fresh environment setup
import sys, subprocess, pkgutil

required = ["transformers", "pillow", "requests", "matplotlib", "torch", "torchvision"]
missing = [p for p in required if pkgutil.find_loader(p) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])
else:
    print("All required packages already installed.")


In [ ]:
import os
import math
import textwrap
import requests
from io import BytesIO

import torch
import matplotlib.pyplot as plt
from PIL import Image

from transformers import (
    CLIPProcessor,
    CLIPModel,
    BlipProcessor,
    BlipForConditionalGeneration,
    BlipForQuestionAnswering,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Small helper for CPU mode
GEN_MAX_NEW_TOKENS = 20 if device == "cpu" else 30


## 0. Install dependencies

Run this cell once. If your environment already has these packages, you can skip it.

In [ ]:
# Uncomment if needed:
# %pip install -q transformers pillow requests matplotlib torch torchvision

## 1. Imports and setup

In [ ]:
import os
import math
import textwrap
import requests
from io import BytesIO

import torch
import matplotlib.pyplot as plt
from PIL import Image

from transformers import (
    CLIPProcessor,
    CLIPModel,
    BlipProcessor,
    BlipForConditionalGeneration,
    BlipForQuestionAnswering,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Small helper for CPU mode
GEN_MAX_NEW_TOKENS = 20 if device == "cpu" else 30


## 2. Load one demo image

You can use the default image below or replace the URL with another image.

In [ ]:
IMAGE_URL = "https://images.unsplash.com/photo-1517841905240-472988babdf9?auto=format&fit=crop&w=900&q=80"

def load_image_from_url(url: str) -> Image.Image:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return Image.open(BytesIO(response.content)).convert("RGB")

image = load_image_from_url(IMAGE_URL)
image

## 3. CLIP: shared image-text embedding space

CLIP does **not generate text**.  
Instead, it maps images and text into a **shared vector space**.

That lets us answer questions like:

> Which text description best matches this image?

### Worksheet prompt
Write down:
1. Which text prompt got the highest CLIP score?
2. Which prompt was second highest?
3. Did the ranking match your intuition?


**Your notes:**  
-  
-  
-  


In [ ]:
clip_model_name = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
clip_model = CLIPModel.from_pretrained(clip_model_name).to(device)
clip_model.eval()

candidate_texts = [
    "a girl running outdoors",
    "a cat sitting on a sofa",
    "a person riding a bicycle",
    "a close-up portrait of a dog",
    "an airplane on a runway",
]

inputs = clip_processor(
    text=candidate_texts,
    images=image,
    return_tensors="pt",
    padding=True,
).to(device)

with torch.no_grad():
    outputs = clip_model(**inputs)
    logits_per_image = outputs.logits_per_image
    probs = logits_per_image.softmax(dim=1).cpu().numpy()[0]

for text, p in sorted(zip(candidate_texts, probs), key=lambda x: -x[1]):
    print(f"{p:.3f}  {text}")

### Exercise 1
Try changing the candidate texts:
- make them more specific
- make two prompts very similar
- add one clearly wrong option

**Question:** How sensitive is CLIP to wording?

## 4. Inspect CLIP image embeddings

Now we look inside the image encoder.

A ViT-style encoder splits the image into **patches** and maps them to vectors.

In [ ]:
with torch.no_grad():
    pixel_values = clip_processor(images=image, return_tensors="pt")["pixel_values"].to(device)
    vision_outputs = clip_model.vision_model(pixel_values=pixel_values)

last_hidden_state = vision_outputs.last_hidden_state
pooler_output = vision_outputs.pooler_output

print("pixel_values shape:", tuple(pixel_values.shape))
print("vision last_hidden_state shape:", tuple(last_hidden_state.shape))
print("vision pooler_output shape:", tuple(pooler_output.shape))

### What do these shapes mean?

Typical interpretation:
- `pixel_values`: batch, channels, height, width
- `last_hidden_state`: batch, number_of_tokens, hidden_dim
- `pooler_output`: one summary vector for the full image

For CLIP ViT-B/32, the token sequence usually contains:
- 1 global/class token
- patch tokens for the image

This is the key bridge to multimodal LLMs:

> the image becomes a **sequence of vectors**, similar in spirit to text tokens.

## 5. Visualize image patches

This is a simple way to make the idea of patchification concrete.

In [ ]:
def show_image_patches(img: Image.Image, patch_size: int = 32, max_cols: int = 6):
    w, h = img.size
    cols = min(max_cols, math.ceil(w / patch_size))
    rows = math.ceil(h / patch_size)

    fig = plt.figure(figsize=(12, 2 * rows))
    patch_idx = 1
    for y in range(0, h, patch_size):
        for x in range(0, w, patch_size):
            patch = img.crop((x, y, min(x + patch_size, w), min(y + patch_size, h)))
            ax = fig.add_subplot(rows, cols, patch_idx)
            ax.imshow(patch)
            ax.axis("off")
            ax.set_title(f"{x},{y}", fontsize=8)
            patch_idx += 1
            if patch_idx > rows * cols:
                break
        if patch_idx > rows * cols:
            break
    plt.tight_layout()
    plt.show()

show_image_patches(image, patch_size=64, max_cols=6)

## 6. Image captioning with BLIP

Unlike CLIP, BLIP is a **generative vision-language model**.

It can produce text from an image, for example a caption.

### Worksheet prompt
After generating the caption:
- Is it accurate?
- Is it too vague?
- What visual details are missing?


**Your notes:**  
-  
-  
-  


In [ ]:
caption_model_name = "Salesforce/blip-image-captioning-base"
caption_processor = BlipProcessor.from_pretrained(caption_model_name)
caption_model = BlipForConditionalGeneration.from_pretrained(caption_model_name).to(device)
caption_model.eval()

inputs = caption_processor(images=image, return_tensors="pt").to(device)

with torch.no_grad():
    generated_ids = caption_model.generate(**inputs, max_new_tokens=GEN_MAX_NEW_TOKENS)

caption = caption_processor.decode(generated_ids[0], skip_special_tokens=True)
print("Caption:", caption)

### Prompted captioning

Some captioning models respond differently when given a text prefix.

In [ ]:
prompt = "a photography of"
inputs = caption_processor(images=image, text=prompt, return_tensors="pt").to(device)

with torch.no_grad():
    generated_ids = caption_model.generate(**inputs, max_new_tokens=GEN_MAX_NEW_TOKENS)

caption_prompted = caption_processor.decode(generated_ids[0], skip_special_tokens=True)
print("Prompted caption:", caption_prompted)

### Exercise 2
Try a few different prefixes:
- `a close-up of`
- `an image of`
- `this picture shows`

**Question:** How much can prompting change the generated description?

## 7. Visual Question Answering (VQA)

Now we ask questions *about* the image.

This is closer to what students think of as a multimodal assistant.

### Worksheet prompt
Test at least three question types:
- object recognition
- counting
- reasoning

For each, note whether the answer was:
- correct
- partly correct
- clearly wrong


**Your notes:**  
-  
-  
-  


In [ ]:
vqa_model_name = "Salesforce/blip-vqa-base"
vqa_processor = BlipProcessor.from_pretrained(vqa_model_name)
vqa_model = BlipForQuestionAnswering.from_pretrained(vqa_model_name).to(device)
vqa_model.eval()

def ask_vqa(img: Image.Image, question: str, max_new_tokens: int = GEN_MAX_NEW_TOKENS) -> str:
    inputs = vqa_processor(images=img, text=question, return_tensors="pt").to(device)
    with torch.no_grad():
        out = vqa_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return vqa_processor.decode(out[0], skip_special_tokens=True)

questions = [
    "What animal is shown?",
    "Where is the animal?",
    "Is this indoors or outdoors?",
    "What is the main subject of the image?",
]

for q in questions:
    ans = ask_vqa(image, q)
    print(f"Q: {q}\nA: {ans}\n")

## 8. Test strengths and weaknesses

Now let's probe the model.

Try questions from different categories:
- recognition: *What animal is this?*
- attributes: *What color is the fur?*
- location: *Is this indoors or outdoors?*
- counting: *How many dogs are visible?*
- reasoning: *What might happen next?*
- OCR: *What text is written in the image?* (often weak if there is small text)

Use the cell below.

In [ ]:
custom_questions = [
    "How many animals are visible?",
    "What is likely happening in the scene?",
    "Is the image realistic or AI-generated?",
]

for q in custom_questions:
    ans = ask_vqa(image, q)
    print(f"Q: {q}\nA: {ans}\n")

### Exercise 3
Find one question the model answers well, and one it answers poorly.

Then discuss:
1. Was the failure due to **vision**?
2. Was it due to **language**?
3. Was it due to **reasoning**?

## 9. Inspect BLIP internals

Let's inspect the tensor shapes going into the generative model.

In [ ]:
sample_inputs = caption_processor(images=image, return_tensors="pt").to(device)
for k, v in sample_inputs.items():
    print(k, tuple(v.shape))

In [ ]:
# Some model internals are accessible through submodules.
# Exact names can differ slightly across model versions,
# so we keep this inspection lightweight and robust.

print(caption_model.__class__.__name__)
print("\nTop-level modules:")
for name, module in list(caption_model.named_children()):
    print("-", name, "->", module.__class__.__name__)

## 10. Compare CLIP and BLIP

This is the conceptual contrast you want students to leave with:

| Model | Main idea | Output |
|---|---|---|
| CLIP | Shared image-text embedding space | similarity scores |
| BLIP captioning | Generate text from image | caption |
| BLIP VQA | Answer questions about image | answer |

In lecture language:
- **CLIP** aligns modalities
- **generative VLMs** use those aligned representations for language generation

## Short written reflection

Complete these sentences:

- **CLIP is good for** ...
- **BLIP is good for** ...
- **A limitation of CLIP is** ...
- **A limitation of BLIP/VQA is** ...


**Your notes:**  
-  
-  
-  


## 11. Student mini-task

Choose **one new image** and do all three:
1. CLIP prompt matching
2. BLIP captioning
3. BLIP question answering

Then write 3–4 sentences answering:
- What did the models do well?
- What did they do poorly?
- Which model felt more "intelligent", and why?

In [ ]:
# Replace with any image URL you want to test.
NEW_IMAGE_URL = "https://images.unsplash.com/photo-1507146426996-ef05306b995a?auto=format&fit=crop&w=900&q=80"

new_image = load_image_from_url(NEW_IMAGE_URL)
new_image

In [ ]:
# --- CLIP on the new image ---
new_candidate_texts = [
    "a dog outdoors",
    "a cat indoors",
    "a car parked on a street",
    "a child playing football",
]

inputs = clip_processor(
    text=new_candidate_texts,
    images=new_image,
    return_tensors="pt",
    padding=True,
).to(device)

with torch.no_grad():
    outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1).cpu().numpy()[0]

print("CLIP ranking:")
for text, p in sorted(zip(new_candidate_texts, probs), key=lambda x: -x[1]):
    print(f"{p:.3f}  {text}")

# --- BLIP captioning ---
inputs = caption_processor(images=new_image, return_tensors="pt").to(device)
with torch.no_grad():
    generated_ids = caption_model.generate(**inputs, max_new_tokens=GEN_MAX_NEW_TOKENS)
print("\nCaption:", caption_processor.decode(generated_ids[0], skip_special_tokens=True))

# --- BLIP VQA ---
for q in ["What is in the image?", "Where is the animal?", "How many animals are visible?"]:
    print(f"\nQ: {q}")
    print("A:", ask_vqa(new_image, q))

## 12. Optional extension for stronger hardware

If you have a GPU machine, try a chat-style multimodal model such as **LLaVA**.
That lets students see a system that behaves more like a visual assistant.

Suggested model families:
- `llava-hf/llava-1.5-7b-hf`
- smaller community variants if memory is limited

These models are much heavier than BLIP, so they are better as an optional demo than a mandatory classroom exercise.

## 12a. Extended demo: LLaVA as a stronger multimodal assistant

This section is intended for a **live demo** or for student groups with access to a **GPU runtime**.

LLaVA is closer to the modern chat-style multimodal assistants students know from practice.  
Compared with BLIP, it is usually better at:
- following longer instructions
- answering multi-part questions
- producing richer descriptions
- handling conversational prompts

It is also much heavier, so this section is optional.

### Suggested runtime
- Colab with GPU
- T4 or better recommended
- Not ideal for CPU-only classroom runs


### Worksheet prompt
Before running LLaVA, make a prediction:

1. In what ways do you expect LLaVA to outperform BLIP?
2. What kinds of errors do you think it may still make?
3. Which tasks do you think remain difficult even for stronger multimodal models?


**Your notes:**  
-  
-  
-  


In [ ]:
# Optional install for LLaVA demo
# Run this only if you want the extended demo.

# %pip install -q transformers accelerate sentencepiece

In [ ]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration

LLAVA_MODEL_NAME = "llava-hf/llava-1.5-7b-hf"

use_llava_demo = device == "cuda"  # safer default
print("Run LLaVA demo:", use_llava_demo)

if use_llava_demo:
    llava_processor = AutoProcessor.from_pretrained(LLAVA_MODEL_NAME)
    llava_model = LlavaForConditionalGeneration.from_pretrained(
        LLAVA_MODEL_NAME,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        low_cpu_mem_usage=True,
    ).to(device)
    llava_model.eval()
    print("LLaVA loaded.")
else:
    print("Skipping by default because no GPU was detected.")

### A helper function for chat-style prompting

LLaVA takes both an image and a text prompt.  
We wrap that in a small helper function.


In [ ]:
def ask_llava(img, prompt, max_new_tokens=80):
    if not use_llava_demo:
        return "LLaVA demo not active. Use a GPU runtime to run this section."

    conversation = f"USER: <image>\n{prompt}\nASSISTANT:"
    inputs = llava_processor(text=conversation, images=img, return_tensors="pt").to(device)

    with torch.no_grad():
        output = llava_model.generate(**inputs, max_new_tokens=max_new_tokens)

    decoded = llava_processor.decode(output[0], skip_special_tokens=True)
    return decoded

## 12b. Compare BLIP and LLaVA on the same image

Use the same image as before.  
Try richer prompts than in the BLIP section.


In [ ]:
llava_prompts = [
    "Describe this image in detail.",
    "List the main objects in the scene.",
    "What is the most likely setting or environment?",
]

for p in llava_prompts:
    print(f"Prompt: {p}")
    print(ask_llava(image, p))
    print("\n" + "-" * 80 + "\n")

### Worksheet prompt
Compare the BLIP output and the LLaVA output.

- Which one is more detailed?
- Which one follows the instruction better?
- Which one sounds more like a conversational assistant?
- Did LLaVA introduce any hallucinated details?


**Your notes:**  
-  
-  
-  


## 12c. Test more powerful behaviors

Now try prompts that go beyond short captioning.
These are the kinds of capabilities where a stronger multimodal assistant is often more interesting.


In [ ]:
advanced_prompts = [
    "Write a short alt-text description for a visually impaired user.",
    "Describe the scene in 3 bullet points.",
    "What details in the image support your interpretation?",
    "What might happen next in this scene? Give a cautious answer.",
    "Is there anything uncertain or ambiguous in the image?",
]

for p in advanced_prompts:
    print(f"Prompt: {p}")
    print(ask_llava(image, p))
    print("\n" + "=" * 80 + "\n")

### Worksheet prompt
Which of these prompt types seemed strongest?
- detailed description
- structured formatting
- cautious reasoning
- explanation of evidence
- uncertainty awareness

Did the model actually ground its answer in the image?


**Your notes:**  
-  
-  
-  


## 12d. Stress-test LLaVA

Here we intentionally ask harder questions.
Modern multimodal models are better than BLIP, but they are still imperfect.


In [ ]:
stress_test_prompts = [
    "How many visible objects of the main type are there? Answer carefully.",
    "Read any text visible in the image.",
    "Describe the exact spatial relationship between the main objects.",
    "Could this image be AI-generated? Explain your reasoning briefly.",
    "What can you infer with high confidence, and what remains uncertain?",
]

for p in stress_test_prompts:
    print(f"Prompt: {p}")
    print(ask_llava(image, p))
    print("\n" + "=" * 80 + "\n")

### Worksheet prompt
Find one case where LLaVA is clearly better than BLIP, and one case where it still fails.

For each case, write:
1. the prompt
2. the answer
3. why you think the model succeeded or failed


**Your notes:**  
-  
-  
-  


## 12e. Try your own image and prompt

For a mini-project, test one new image with:
- one simple recognition prompt
- one reasoning prompt
- one stress-test prompt

Then compare CLIP, BLIP, and LLaVA qualitatively.


In [ ]:
# Replace with your own image URL if desired
EXTRA_IMAGE_URL = "https://images.unsplash.com/photo-1518717758536-85ae29035b6d?auto=format&fit=crop&w=900&q=80"
extra_image = load_image_from_url(EXTRA_IMAGE_URL)
extra_image

In [ ]:
custom_llava_prompts = [
    "What is in this image?",
    "What is the subject doing, and how do you know?",
    "What might a weaker vision-language model miss here?",
]

for p in custom_llava_prompts:
    print(f"Prompt: {p}")
    print(ask_llava(extra_image, p))
    print("\n" + "-" * 80 + "\n")

## 12f. Discussion

Use this section to connect the demo back to the lecture:

- **CLIP** aligns image and text representations.
- **BLIP** generates captions and short answers from images.
- **LLaVA** behaves more like a general multimodal assistant.

A useful interpretation is:

- BLIP = strong task-specific vision-language model
- LLaVA = more flexible multimodal chat model


### Final worksheet reflection for the LLaVA extension

Complete these sentences:

- **The biggest advantage of LLaVA over BLIP was** ...
- **A limitation that remained even with LLaVA was** ...
- **The most convincing example of multimodal reasoning in this notebook was** ...
- **The most important open challenge for these models is** ...


**Your notes:**  
-  
-  
-  


## 13. Takeaway

A compact summary of the notebook:

```text
CLIP: image + text -> shared embedding space
BLIP: image -> text generation
VQA: image + question -> answer
```

The core lesson is:

> Multimodal models work by converting images into vector representations that can interact with language representations.